[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Adding and Changing Columns &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})


def print_warnings(caught):
    """Print each caught warning as its category and the first line of its message."""
    for warning in caught:
        print(f"{warning.category.__name__}: {str(warning.message).strip().splitlines()[0]}")


print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Make `table` again with `readings.copy()`, add a column `kelvin` holding each reading plus 273.15, rounded to two places, and show the first three rows.

In [2]:
table = readings.copy()
table["kelvin"] = (table["celsius"] + 273.15).round(2)

table.head(3)

,station,time,celsius,kelvin
0,Bergen,2025-01-01,-3.6,269.55
1,Oslo,2025-01-01,-3.5,269.65
2,Svalbard,2025-01-01,-14.6,258.55


The rounding hides the floating point digits that adding 273.15 leaves behind.

**2.** Use `assign` to make a table with a column `month`, from `time` with `.dt.month`, and print how many readings fall in July.

In [3]:
with_month = readings.assign(month=readings["time"].dt.month)

print((with_month["month"] == 7).sum())
print("month" in readings.columns)

2976
False


July has 31 days of 24 hours at four stations, 2,976 readings. `readings` itself has no `month` column, because `assign` returned a new table.

**3.** In a copy of `readings`, set `celsius` to `NaN` for Bergen's readings on 1 January with `loc`, and print how many of Bergen's readings are now missing.

In [4]:
table = readings.copy()
new_year = (table["station"] == "Bergen") & (table["time"] < "2025-01-02")
table.loc[new_year, "celsius"] = np.nan

print(table.loc[table["station"] == "Bergen", "celsius"].isna().sum())

24


The mask picks Bergen's 24 hours of 1 January, and `loc` changes only those rows of `celsius`.

**4.** Make a table of Oslo's readings with `.copy()`, add a `fahrenheit` column to it inside `warnings.catch_warnings(record=True)`, and print how many warnings were caught.

In [5]:
oslo = readings[readings["station"] == "Oslo"].copy()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    oslo["fahrenheit"] = oslo["celsius"] * 9 / 5 + 32

print(len(caught), "warnings")

0 warnings


`.copy()` makes `oslo` a table of its own, so pandas has no reason to warn.

**5.** Make a new table with `celsius` renamed to `temp_c`, and check that `readings` still has a column called `celsius`.

In [6]:
renamed = readings.rename(columns={"celsius": "temp_c"})

print(renamed.columns.tolist())
print("celsius" in readings.columns)

['station', 'time', 'temp_c']
True


`rename` returns a new table and leaves `readings` with its own names.

---

&#8592; **Back to:** [Adding and Changing Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/11-adding-and-changing-columns.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
